In [1]:
import sys
from pathlib import Path

# Walk upward from the current working directory until we find the
# repository root. We define the repo root as the folder that contains
# both `src/` (our Python package) and `data/` (our datasets).
#
# This is necessary because Jupyter in VS Code / Codespaces often runs
# with cwd = /.../notebooks instead of the project root.

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src").exists() and (p / "data").exists():
            return p
    raise RuntimeError(f"Could not find repo root above {start}")

# Determine the true repository root regardless of where the notebook
# kernel was launched from

REPO_ROOT = find_repo_root(Path.cwd())

# Add the repo root to Python's import search path so that
# `import src.io` works inside notebooks

sys.path.insert(0, str(REPO_ROOT))

# Build an absolute path to the raw data directory so we never rely on
# fragile relative paths like "data/raw"

RAW_DIR = REPO_ROOT / "data" / "raw"
RAW_DIR = RAW_DIR.resolve()

# Now that paths and imports are stable, we can safely import libraries
# and our project code

import pandas as pd
from src.io import ingest_raw_csvs

# Display for sanity checking
REPO_ROOT, RAW_DIR

(PosixPath('/workspaces/protected-bike-lanes-ridership'),
 PosixPath('/workspaces/protected-bike-lanes-ridership/data/raw'))

In [2]:
import os

# Show the actual current working directory of the notebook kernel.
# In VS Code / Codespaces this is usually /.../notebooks instead of the repo root.
print("cwd:", os.getcwd())

# Show the repository root we detected via the bootstrap logic.
# This should point to /workspaces/protected-bike-lanes-ridership
print("repo_root:", REPO_ROOT)

# Further sanity check that the src/ package is actually present at the repo root.
# If this is False, imports like `from src.io import ...` will fail.
print("src exists:", (REPO_ROOT / "src").exists())

# Further sanity check that the data/ directory is present.
# If this is False, RAW_DIR construction is broken.
print("data exists:", (REPO_ROOT / "data").exists())

# Show the first entry on Python's module search path.
# This should be the repo root, meaning Python can find `src/` as a package.
print("sys.path[0]:", sys.path[0])

cwd: /workspaces/protected-bike-lanes-ridership/notebooks
repo_root: /workspaces/protected-bike-lanes-ridership
src exists: True
data exists: True
sys.path[0]: /workspaces/protected-bike-lanes-ridership


In [3]:
# Run the raw CSV ingest pipeline against the raw data directory.
# This reads every CSV under data/raw/, applies any safe parsing logic,
# and combines them into a single pandas DataFrame.
result = ingest_raw_csvs(RAW_DIR)

# The unified dataframe of all counter data
df = result.df

# Basic sanity checks so we know we didn't silently fail
print("Rows:", len(df))                      # total number of records loaded
print("Columns:", len(df.columns))           # how wide the dataset is
print("Files read:", len(result.files_read)) # how many CSVs were successfully ingested
print("Files failed:", len(result.files_failed))  # how many CSVs could not be read

# Peek at the first few rows to visually confirm the structure
df.head()

/workspaces/protected-bike-lanes-ridership/src/io.py:40: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, encoding="utf-8")


Rows: 838202
Columns: 38
Files read: 10
Files failed: 0


,date,2nd_ave_cycletrack,bicyclists_northbound,bicyclists_southbound,scooterist_northbound,scooterist_southbound,source_file,39th_ave_ne_greenway_at_ne_62nd_st_total,north,south,...,pedestrian_west,pedestrian_east,bike_east,bike_west,scooter_east,scooter_west,nw_58th_st_greenway_st_22nd_ave_nw_total,east,west,spokane_st._bridge_total
0,2015 Jan 01 12:00:00 AM,3.0,0.0,3.0,NaN,NaN,/workspaces/protected-bike-lanes-ridership/dat...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015 Jan 01 01:00:00 AM,9.0,0.0,9.0,NaN,NaN,/workspaces/protected-bike-lanes-ridership/dat...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015 Jan 01 02:00:00 AM,2.0,0.0,2.0,NaN,NaN,/workspaces/protected-bike-lanes-ridership/dat...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2015 Jan 01 03:00:00 AM,0.0,0.0,0.0,NaN,NaN,/workspaces/protected-bike-lanes-ridership/dat...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2015 Jan 01 04:00:00 AM,0.0,0.0,0.0,NaN,NaN,/workspaces/protected-bike-lanes-ridership/dat...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# List all column names in sorted order so we can see the full schema
# and spot things like multiple date fields, direction columns, or
# inconsistently named variables across files.
pd.Series(df.columns).sort_values()

1                                    2nd_ave_cycletrack
7              39th_ave_ne_greenway_at_ne_62nd_st_total
13                           bgt_north_of_ne_70th_total
22                                 bicyclist_northbound
23                                 bicyclist_southbound
2                                 bicyclists_northbound
3                                 bicyclists_southbound
30                                            bike_east
16                                           bike_north
17                                           bike_south
31                                            bike_west
10       broadway_cycle_track_north_of_e_union_st_total
18              chief_sealth_trl_north_of_thistle_total
0                                                  date
35                                                 east
19       elliott_bay_trail_in_myrtle_edwards_park_total
26    fremont_bridge_sidewalks,_south_of_n_34th_st_c...
25    fremont_bridge_sidewalks,_south_of_n_34th_

In [5]:
# Identify any columns that look like date or time fields.
# Seattle data often includes multiple timestamp columns, so this
# helps us see our candidates before deciding which one to use
# for daily aggregation.
[c for c in df.columns if "date" in c or "time" in c]

['date']

In [6]:
# Compute the fraction of missing values in each column and show
# the 20 most incomplete fields. This tells us which columns are
# mostly empty metadata and which ones are reliable enough to use
# for analysis and modeli
df.isna().mean().sort_values(ascending=False).head(20)

scooter_east                                      0.984050
scooter_west                                      0.984050
scooterist_northbound                             0.984024
scooterist_southbound                             0.984024
chief_sealth_trl_north_of_thistle_total           0.965198
39th_ave_ne_greenway_at_ne_62nd_st_total          0.953877
south                                             0.953877
north                                             0.953877
sb                                                0.924185
nb                                                0.924185
broadway_cycle_track_north_of_e_union_st_total    0.924185
nw_58th_st_greenway_st_22nd_ave_nw_total          0.912268
ped_north                                         0.910820
ped_south                                         0.910820
pedestrian_west                                   0.904382
pedestrian_east                                   0.904382
bike_west                                         0.9026